# 🛡️ TorchShield: Interactive Guide to Zero-Pickle AI Checkpoints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/csharikrishna/TorchShield/blob/master/notebooks/TorchShield_Interactive_Walkthrough.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/csharikrishna/TorchShield)
[![Showcase Site](https://img.shields.io/badge/Docs-Showcase%20Site-emerald)](https://csharikrishna.github.io/TorchShield/)
[![Version](https://img.shields.io/badge/Version-v0.7.0-indigo)](https://github.com/csharikrishna/TorchShield/releases/tag/v0.7.0)

Welcome to the official interactive walkthrough for **TorchShield** (`v0.7.0`).

### Why TorchShield?
In modern deep learning workflows, model and training state checkpointing is critical. However, current practices introduce severe vulnerabilities:
1. **The Insecure Pickle Vulnerability:** Standard PyTorch checkpoints (`torch.save` / `torch.load`) serialize Python objects using `pickle`. Deserializing untrusted checkpoints can execute arbitrary Python bytecode and shell commands (Remote Code Execution, CVE-2024-3568, etc.).
2. **The Hugging Face Dilemma:** While Hugging Face solved model weight serialization with `safetensors`, their `Trainer` **still saves `optimizer.pt`, `scheduler.pt`, and `rng_state.pth` as raw pickle files**.
3. **Weight Tampering & Bit-Rot:** Traditional formats lack authenticated streaming encryption (AEAD) and digital signatures. An attacker can silently backdoor weight tensors without raising any integrity errors.
4. **Replay & Rollback Attacks:** In continuous training, adversaries can replay an older, vulnerable, or degraded checkpoint generation.

**TorchShield provides an enterprise-grade, pickle-free, cryptographically authenticated checkpoint platform for PyTorch and Hugging Face.**

---

### Notebook Outline
1. **The Insecure Pickle Problem** (Demonstrating how `pickle` allows arbitrary code execution)
2. **Installing TorchShield** (Setting up the environment)
3. **Basic PyTorch Checkpoint Hardening** (Saving model, optimizer, scheduler, and RNGs)
4. **Tamper Detection & Fail-Closed Defense** (Simulating an active bit-flip attack)
5. **Bit-for-Bit Mathematical Equivalence** (Verifying exact float reproducibility)
6. **Hugging Face Trainer Integration** (Safely purging insecure `.pt` pickle files)
7. **Monotonic Anti-Rollback Freshness** (Preventing state replays with SQLite ledgers)
8. **AWS KMS Cloud Key Provider** (Enterprise envelope encryption architecture)


---
## 1. Setup & Installation
Let's install `torchshield` and dependencies directly into this Colab environment.


In [ ]:
# Install TorchShield directly from the official GitHub repository
!pip install --quiet git+https://github.com/csharikrishna/TorchShield.git
!pip install --quiet transformers accelerate

import torchshield
import torch
print(f'✓ TorchShield version: {torchshield.__version__}')
print(f'✓ PyTorch version: {torch.__version__}')


---
## 2. The Insecure Pickle Problem
When standard PyTorch code calls `torch.save(model.state_dict(), 'checkpoint.pt')`, it uses Python's standard `pickle` engine under the hood.

`pickle` is a stack-based virtual machine designed for general-purpose object serialization, **not security**. It includes opcodes that instantiate arbitrary classes and invoke system calls upon deserialization.

Let's inspect how an adversarial payload operates in raw Python pickle:


In [ ]:
import pickle
import os

# Craft an adversarial object that executes a shell command when deserialized
class MaliciousPayload:
    def __reduce__(self):
        # The tuple returns (callable, arguments) executed by the unpickler VM:
        return (os.system, ('echo "[EXPLOIT DETECTED] Malicious shell command executed during unpickling!"',))

# Serialize the payload into raw bytecode
exploit_bytes = pickle.dumps(MaliciousPayload())
print(f'Malicious bytecode size: {len(exploit_bytes)} bytes')
print(f'Bytecode representation: {exploit_bytes}')

# Demonstrating what happens during standard deserialization:
print('\nSimulating unpickling in an unprotected pipeline:')
pickle.loads(exploit_bytes)


> ⚠️ **The Security Risk:** Anyone sharing a `.pt` or `.bin` checkpoint file online could embed arbitrary bytecode that runs with full system permissions the moment someone calls `torch.load(..., weights_only=False)`.
>
> **How TorchShield Solves This:** TorchShield is **100% pickle-free**. It never invokes `pickle.load` or `torch.load`. Tensor parameters are serialized as contiguous IEEE 754 binary buffers, and state metadata is encoded using strictly allowlisted, deterministic canonical JSON (TorchShield Canonical JSON v1) manifests. Executable bytecode cannot be evaluated.


---
## 3. Basic PyTorch Checkpoint Hardening
Let's define a PyTorch neural network, perform training steps, and save the complete training state using TorchShield's streaming AEAD encryption (AES-256-GCM).


In [ ]:
import secrets
import torch.nn as nn
from torchshield import (
    save_training_checkpoint,
    load_training_checkpoint,
    inspect_training_checkpoint,
    TrainingProgress,
)

# Define a standard neural network
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )
    def forward(self, x):
        return self.net(x)

model = Classifier()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# Simulate 5 training steps
torch.manual_seed(42)
for step in range(1, 6):
    inputs = torch.randn(8, 64)
    targets = torch.randint(0, 10, (8,))
    criterion = nn.CrossEntropyLoss()
    loss = criterion(model(inputs), targets)
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    print(f'Step {step} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')


### Saving the Checkpoint
We save the checkpoint using `save_training_checkpoint()`. This encapsulates:
- Model weights & layer parameters (`state_dict`)
- Optimizer first & second moment tensors (`exp_avg`, `exp_avg_sq`)
- Learning rate scheduler states
- Complete RNG states (Python `random`, NumPy, PyTorch CPU, and PyTorch CUDA)
- Cryptographic sequential counter nonces (`P = 0` collision risk) under AES-256-GCM


In [ ]:
# Generate a secure 256-bit (32-byte) key (can come from AWS KMS in production)
encryption_key = secrets.token_bytes(32)

# Save complete training state
save_training_checkpoint(
    output_path='checkpoint_step5.shield',
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    progress=TrainingProgress(global_step=5, epoch=1, loss=loss.item()),
    key=encryption_key,
)

file_size = os.path.getsize('checkpoint_step5.shield')
print(f'✓ Checkpoint securely written to: checkpoint_step5.shield ({file_size:,} bytes)')


### Inspecting Checkpoint Metadata Safely
You can inspect the contents and training metadata of any `.shield` container without loading tensors into GPU memory or executing code:


In [ ]:
# Inspect metadata using inspect_training_checkpoint
meta = inspect_training_checkpoint('checkpoint_step5.shield', key=encryption_key)

print('Authenticated Checkpoint Metadata:')
print(f'  • Global Step:    {meta.get("global_step")}')
print(f'  • Epoch:          {meta.get("epoch")}')
print(f'  • Loss:           {meta.get("loss"):.4f}')
print(f'  • Authenticated:  {meta.get("authenticated")}')


---
## 4. Security Demonstration: Active Tamper Detection & Fail-Closed Defense
What happens if an adversary modifies a model weight byte on disk (or disk bit-rot corrupts a tensor)?

Let's intentionally inject a bit-flip into the encrypted chunk envelope on disk.


In [ ]:
# Open the file and flip 1 byte in the weight chunk envelope
with open('checkpoint_step5.shield', 'r+b') as f:
    f.seek(64) # Seek into the encrypted chunk payload
    original_byte = f.read(1)[0]
    corrupted_byte = original_byte ^ 0xFF # Invert bits
    f.seek(64)
    f.write(bytes([corrupted_byte]))

print(f'Injected tampering at byte offset 64: 0x{original_byte:02x} -> 0x{corrupted_byte:02x}')


Now let's attempt to load this tampered checkpoint into a fresh model instance:


In [ ]:
fresh_model = Classifier()
fresh_optimizer = torch.optim.AdamW(fresh_model.parameters(), lr=1e-3)

try:
    load_training_checkpoint(
        input_path='checkpoint_step5.shield',
        model=fresh_model,
        optimizer=fresh_optimizer,
        key=encryption_key,
    )
    print('❌ ERROR: Tampered checkpoint loaded! (Should never happen)')
except Exception as e:
    print(f'🛡️ DEFENSE TRIGGERED! Caught expected exception:')
    print(f'   {type(e).__name__}: {e}')
    print('\n✓ Security Guarantee: Memory allocation aborted. All staging buffers scrubbed.')


---
## 5. Mathematical Bit-for-Bit Training Equivalence
Does zero-pickle serialization affect numerical precision or convergence?

Let's test this mathematically:
- **Pipeline A:** Trains continuously for 10 steps without stopping.
- **Pipeline B:** Trains for 5 steps, saves a `.shield` checkpoint, simulates a process restart, loads the checkpoint, and continues training for steps 6 to 10.

We verify that every float parameter tensor matches identically: $\text{diff} = 0.0$.


In [ ]:
def create_pipeline(seed=1337):
    torch.manual_seed(seed)
    m = Classifier()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    return m, opt

# Pipeline A: Uninterrupted baseline (10 steps)
mA, optA = create_pipeline(1337)
torch.manual_seed(999)
data_batches = [torch.randn(8, 64) for _ in range(10)]
target_batches = [torch.randint(0, 10, (8,)) for _ in range(10)]

for i in range(10):
    lossA = nn.CrossEntropyLoss()(mA(data_batches[i]), target_batches[i])
    lossA.backward()
    optA.step()
    optA.zero_grad()

# Pipeline B: Interrupted and resumed at step 5
mB, optB = create_pipeline(1337)
equiv_key = secrets.token_bytes(32)

for i in range(5):
    lossB = nn.CrossEntropyLoss()(mB(data_batches[i]), target_batches[i])
    lossB.backward()
    optB.step()
    optB.zero_grad()

# Save checkpoint at step 5
save_training_checkpoint(
    output_path='equiv_checkpoint.shield',
    model=mB,
    optimizer=optB,
    progress=TrainingProgress(global_step=5),
    key=equiv_key,
)

# Simulate fresh process restart with fresh zero-initialized model
m_resumed, opt_resumed = create_pipeline(0)
load_training_checkpoint(
    input_path='equiv_checkpoint.shield',
    model=m_resumed,
    optimizer=opt_resumed,
    key=equiv_key,
)

# Resume training steps 6 to 10
for i in range(5, 10):
    loss_resumed = nn.CrossEntropyLoss()(m_resumed(data_batches[i]), target_batches[i])
    loss_resumed.backward()
    opt_resumed.step()
    opt_resumed.zero_grad()

# Check exact bitwise equality parameter by parameter
all_match = True
print('Parameter-by-Parameter Bitwise Comparison:')
for (nameA, pA), (nameB, pB) in zip(mA.named_parameters(), m_resumed.named_parameters()):
    match = torch.equal(pA, pB)
    max_diff = torch.max(torch.abs(pA - pB)).item()
    print(f'  ✓ {nameA:25} | Equal: {match} | Max Diff: {max_diff}')
    if not match:
        all_match = False

print(f'\nContinuous Final Loss: {lossA.item():.8f}')
print(f'Resumed Final Loss:    {loss_resumed.item():.8f}')
print(f'🎉 100% Bit-for-Bit Mathematical Equivalence: {all_match}')


---
## 6. Hugging Face Trainer Integration
When using Hugging Face's `Trainer`:
- Model weights are saved with `safetensors` (Safe)
- **Optimizer states** are saved as `optimizer.pt` (Unsafe Python `pickle`)
- **Scheduler states** are saved as `scheduler.pt` (Unsafe Python `pickle`)
- **RNG states** are saved as `rng_state.pth` (Unsafe Python `pickle`)

`TorchShieldTrainerCallback` hooks into `Trainer` during `on_save` to write a secure `.shield` container and purge the unsafe `.pt` pickle files automatically!


In [ ]:
from transformers import Trainer, TrainingArguments, TrainerState, TrainerControl
from torchshield import TorchShieldTrainerCallback

hf_key = secrets.token_bytes(32)
callback = TorchShieldTrainerCallback(
    key=hf_key,
    vault_filename='torchshield_checkpoint.shield',
    remove_insecure_pickle=True,  # Automatically cleans up insecure .pt files
)

# Simulate Trainer creating a checkpoint directory
os.makedirs('hf_trainer_output/checkpoint-50', exist_ok=True)

# Simulate insecure pickle files that HF Trainer normally outputs
for fname in ['optimizer.pt', 'scheduler.pt', 'rng_state.pth']:
    with open(f'hf_trainer_output/checkpoint-50/{fname}', 'wb') as f:
        f.write(b'insecure_pickle_bytes')

print('Files in checkpoint directory BEFORE callback:')
print(' ', os.listdir('hf_trainer_output/checkpoint-50'))

# Trigger callback on_save
args = TrainingArguments(output_dir='hf_trainer_output', use_cpu=True)
state = TrainerState(epoch=2.0, global_step=50, log_history=[{'loss': 0.35}])
control = TrainerControl()

callback.on_save(
    args=args,
    state=state,
    control=control,
    model=model,
    optimizer=optimizer,
    lr_scheduler=scheduler,
)

print('\nFiles in checkpoint directory AFTER callback:')
print(' ', os.listdir('hf_trainer_output/checkpoint-50'))
print('✓ Insecure pickle files purged! Replaced with encrypted "torchshield_checkpoint.shield"')


---
## 7. Monotonic Anti-Rollback Freshness Protection
In automated pipelines or continuous learning, an adversary can attempt a **rollback attack** by submitting an older, superseded checkpoint to revert model alignment or exploit unpatched vulnerabilities.

TorchShield provides a local or distributed `SQLiteFreshnessLedger` that enforces monotonic high-water mark validation:


In [ ]:
from torchshield.training_state import SQLiteFreshnessLedger

ledger = SQLiteFreshnessLedger(db_path='freshness_ledger.db')
pipeline_id = 'production-training-run-01'

# Record progressive training generations
ledger.record_generation(pipeline_id, generation=100, checkpoint_digest='sha256:abc123...')
ledger.record_generation(pipeline_id, generation=200, checkpoint_digest='sha256:def456...')

high_water_mark = ledger.get_latest_generation(pipeline_id)
print(f'Current established high-water mark: Generation {high_water_mark}')

# Test candidate checkpoints
test_gen_50 = ledger.validate_generation(pipeline_id, candidate_generation=50)
test_gen_300 = ledger.validate_generation(pipeline_id, candidate_generation=300)

print(f'Candidate Generation 50  (Rollback): Allowed = {test_gen_50} -> REJECTED ❌')
print(f'Candidate Generation 300 (Progress): Allowed = {test_gen_300} -> ACCEPTED ✓')


---
## Summary & Production Deployment

TorchShield gives your AI infrastructure defense-in-depth security:
- **Zero Pickle:** 100% immune to deserialization arbitrary code execution.
- **Streaming AEAD AES-256-GCM:** Deterministic sequential nonces eliminate reuse collisions ($P=0$).
- **Exact Reproducibility:** 100% bit-exact float recovery across model, optimizer, and RNG states.
- **Hugging Face Trainer Ready:** Plugs the optimizer pickle vulnerability.
- **Monotonic Freshness Ledgers:** SQLite anti-rollback protection.
- **Cloud KMS Ready:** AWS KMS envelope encryption.

### Documentation & Community
* 🌐 **Documentation & Interactive Showcase:** [csharikrishna.github.io/TorchShield](https://csharikrishna.github.io/TorchShield/)
* 💻 **GitHub Repository:** [github.com/csharikrishna/TorchShield](https://github.com/csharikrishna/TorchShield)
* 📦 **Release Artifacts & Wheel:** [GitHub Releases v0.7.0](https://github.com/csharikrishna/TorchShield/releases/tag/v0.7.0)
